# 03 - Causal patching (Phase 4), GPU

Activation patching on the 78 (clean, counterfactual) pairs; only token-aligned pairs are used (~51).

* **Experiment A** (main): copy the clean run's residual stream into the CF run at layer L - at the swapped-entity tokens,
  or at the last prompt token - and check whether the model's greedy hop 1 now states the correct bridge entity.
  Includes a donor-entity negative control and an all-positions sanity check.
* **Experiment B**: the old `</hop1>` state transplant, now deterministic (greedy), reported against its baseline.
* The script runs a **hook self-test** first and stops if patching is not taking effect.

Settings: **GPU T4 x1** (or x2), **Internet On**, secret **`HF_TOKEN`** (Llama 3.2 licence). Runtime ~1-1.5 h (pass `--layers 0,4,8,10,11,12,14,20,27` to roughly halve it).
Output: `/kaggle/working/patching_results.zip` -> `summary.md`, `summary.json`, per-pair JSONL.

In [ ]:
# Find the uploaded bundle (any Kaggle Dataset containing lf_common.py) and work on a writable copy.
import os, sys, shutil
from pathlib import Path
hits = sorted(Path("/kaggle/input").rglob("lf_common.py"))
assert hits, ("Bundle not found. Click 'Add Input' (right panel) and add the Kaggle Dataset you created "
              "from linguafranca_bundle.zip.")
src = hits[0].parent
WORK = Path("/kaggle/working/lf")
if not WORK.exists():
    shutil.copytree(src, WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("bundle:", src, "->", WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Hugging Face login (needed for the gated Llama models).
# Kaggle: Add-ons -> Secrets -> add a secret named HF_TOKEN (token from an account that has accepted
# the Llama 3.2 / Llama 3.1 licences on huggingface.co) and switch it on for this notebook.
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN); os.environ["HF_TOKEN"] = HF_TOKEN
    print("Logged in to Hugging Face.")
else:
    print("WARNING: no HF_TOKEN secret - gated Llama models will fail to download (Qwen still works).")

In [ ]:
!python causal_patching.py --data data/canonical/augmented_v2.jsonl --model meta-llama/Llama-3.2-3B-Instruct --out outputs/patching

In [ ]:
print(open("outputs/patching/summary.md").read())

In [ ]:
# Package the results; download the zip from the Output section (right panel) of this notebook.
import shutil
shutil.make_archive("/kaggle/working/patching_results", "zip", "/kaggle/working/lf/outputs/patching")
print("Download /kaggle/working/patching_results.zip from the Output tab.")